# DS1 - Parallel Reduction for Minimum Element using CUDA Python (Numba)

## Dataset Metadata

- Dataset Name: 10 Million Random Number Dataset
- Source: Kaggle
- Kaggle Dataset: https://www.kaggle.com/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml
- Records: 10,000,000
- Features: 50 numerical features
- Data Type: `float32`
- CUDA Operation: Find the global minimum element

## CUDA Task

Input:
- A large floating-point vector from the dataset.

Output:
- Global minimum element.

Parallelization Strategy:
- Tree-based reduction using shared memory.

Important for Kaggle:
- Add the 10 Million Random Number Dataset using Kaggle's "Add Data" panel.
- This notebook does not download the dataset.
- It only reads files from `/kaggle/input`.

Default setting:
- The notebook reduces one 10,000,000-value feature column to match the report table.
- Set `REDUCE_ALL_50_FEATURES = True` to reduce all 500,000,000 values.

In [1]:
# Standard libraries used for paths and timing.
from pathlib import Path
import time

# Numerical, CSV, and GPU libraries available in Kaggle notebooks.
import numpy as np
import pandas as pd
from numba import cuda


# Kaggle automatically mounts added datasets here.
INPUT_ROOT = Path("/kaggle/input")

# Kaggle allows notebook outputs to be written here.
WORKING_ROOT = Path("/kaggle/working")

# Dataset metadata used in error messages and file discovery.
DATASET_URL = "https://www.kaggle.com/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml"
DATASET_SLUG_HINT = "10-million-random-number-dataset-for-ml"


def find_csv_files(root=INPUT_ROOT):
    """Find CSV files from Kaggle input without downloading anything."""
    if not root.exists():
        raise FileNotFoundError(
            "Kaggle input directory was not found. Add the dataset in Kaggle: "
            f"{DATASET_URL}"
        )

    all_csv_files = sorted(root.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    preferred_files = [p for p in all_csv_files if DATASET_SLUG_HINT in str(p).lower()]
    csv_files = preferred_files or all_csv_files

    if not csv_files:
        raise FileNotFoundError(
            "No CSV file was found under /kaggle/input. Add the Kaggle dataset to this notebook."
        )

    print("CSV files detected:")
    for path in csv_files:
        print(f"  {path} ({path.stat().st_size / (1024 ** 3):.2f} GB)")
    return csv_files


def columns_look_like_data(columns):
    """Detect headerless CSV files where the first row was read as column names."""
    labels = pd.Series([str(c) for c in columns])
    parsed = pd.to_numeric(labels, errors="coerce")
    return parsed.notna().mean() > 0.80


def detect_numeric_columns(csv_path, feature_limit=50):
    """Return pandas read options and numeric feature columns."""
    preview = pd.read_csv(csv_path, nrows=256)
    read_kwargs = {}

    # If column labels look numeric, the file probably has no header.
    if columns_look_like_data(preview.columns):
        read_kwargs = {"header": None}
        preview = pd.read_csv(csv_path, nrows=256, header=None)

    numeric_columns = []
    for col in preview.columns:
        numeric = pd.to_numeric(preview[col], errors="coerce")
        if numeric.notna().mean() > 0.95:
            numeric_columns.append(col)

    if not numeric_columns:
        raise ValueError("Could not detect numeric feature columns in the Kaggle CSV.")

    return read_kwargs, numeric_columns[:feature_limit]


def select_dataset_csv():
    """Use the largest CSV from the Kaggle-mounted dataset folder."""
    csv_path = find_csv_files()[0]
    print(f"Using CSV: {csv_path}")
    return csv_path


def dataframe_to_float32(df):
    """Convert a pandas chunk to contiguous float32 values for GPU transfer."""
    arr = df.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32, copy=True)
    if np.isnan(arr).any():
        arr = np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return np.ascontiguousarray(arr)


# Stop early if the Kaggle notebook is not using a GPU.
if not cuda.is_available():
    raise RuntimeError("CUDA is not available. In Kaggle, enable Settings -> Accelerator -> GPU.")

device = cuda.get_current_device()
print(f"CUDA device: {device.name.decode() if isinstance(device.name, bytes) else device.name}")
WORKING_ROOT.mkdir(parents=True, exist_ok=True)

CUDA device: Tesla T4


In [2]:
from numba import cuda, float32


# Number of CUDA threads per block for the reduction kernel.
TPB = 256

# Chunking prevents loading the full 10M x 50 dataset into memory at once.
CHUNK_ROWS = 100_000

# The dataset has 50 numerical features.
FEATURE_LIMIT = 50

# False uses one 10M-value column. True uses all 50 columns, i.e., 500M values.
REDUCE_ALL_50_FEATURES = False

# Use None for all rows, or a smaller number while testing.
ROW_LIMIT = None


@cuda.jit
def reduce_min_kernel(values, partial, n):
    """Reduce a flattened input array to one minimum value per CUDA block."""
    smem = cuda.shared.array(shape=TPB, dtype=float32)
    tid = cuda.threadIdx.x
    start = cuda.blockIdx.x * cuda.blockDim.x * 2 + tid

    # A large float32 value is used as infinity inside the CUDA kernel.
    local_min = np.float32(3.4028234663852886e38)

    # Each thread reads up to two elements, reducing global memory accesses.
    if start < n:
        local_min = values[start]

    second = start + cuda.blockDim.x
    if second < n and values[second] < local_min:
        local_min = values[second]

    # Store the thread-local result in shared memory.
    smem[tid] = local_min
    cuda.syncthreads()

    # Tree-based shared-memory reduction inside the block.
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride and smem[tid + stride] < smem[tid]:
            smem[tid] = smem[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # Thread 0 writes the block minimum.
    if tid == 0:
        partial[cuda.blockIdx.x] = smem[0]


def gpu_reduce_min(values):
    """Launch the reduction kernel repeatedly until one value remains."""
    arr = np.ascontiguousarray(values.ravel(), dtype=np.float32)
    n = arr.size
    if n == 0:
        return float("inf")

    d_in = cuda.to_device(arr)
    current_n = n

    # Each pass reduces the array from N values to roughly N / (2 * TPB) values.
    while current_n > 1:
        blocks = (current_n + (TPB * 2 - 1)) // (TPB * 2)
        d_out = cuda.device_array(blocks, dtype=np.float32)
        reduce_min_kernel[blocks, TPB](d_in, d_out, current_n)
        d_in = d_out
        current_n = blocks

    return float(d_in.copy_to_host()[0])

In [3]:
# Select the Kaggle-mounted CSV dataset.
csv_path = select_dataset_csv()
read_kwargs, numeric_columns = detect_numeric_columns(csv_path, feature_limit=FEATURE_LIMIT)

if REDUCE_ALL_50_FEATURES:
    columns_to_use = numeric_columns
else:
    columns_to_use = [numeric_columns[0]]

print(f"Using {len(columns_to_use)} feature column(s).")
print(f"Chunk rows: {CHUNK_ROWS:,}")

reader = pd.read_csv(
    csv_path,
    usecols=columns_to_use,
    chunksize=CHUNK_ROWS,
    **read_kwargs,
)

rows_done = 0
values_done = 0
gpu_global_min = float("inf")
cpu_global_min = float("inf")
start = time.perf_counter()

for chunk_id, chunk in enumerate(reader, start=1):
    # ROW_LIMIT is useful for quick testing, but None processes the full Kaggle dataset.
    if ROW_LIMIT is not None:
        remaining = ROW_LIMIT - rows_done
        if remaining <= 0:
            break
        if len(chunk) > remaining:
            chunk = chunk.iloc[:remaining]

    values = dataframe_to_float32(chunk)

    # GPU result: parallel reduction using shared memory.
    chunk_gpu_min = gpu_reduce_min(values)

    # CPU result: verification only.
    chunk_cpu_min = float(values.min())

    gpu_global_min = min(gpu_global_min, chunk_gpu_min)
    cpu_global_min = min(cpu_global_min, chunk_cpu_min)
    rows_done += values.shape[0]
    values_done += values.size

    if chunk_id == 1 or chunk_id % 10 == 0:
        print(
            f"Processed chunk {chunk_id:>4}: rows={rows_done:,}, "
            f"values={values_done:,}, current_min={gpu_global_min:.10f}"
        )

elapsed = time.perf_counter() - start

print("\nDS1 Parallel Reduction complete.")
print(f"Rows processed: {rows_done:,}")
print(f"Values processed: {values_done:,}")
print(f"GPU global minimum: {gpu_global_min:.10f}")
print(f"CPU verification minimum: {cpu_global_min:.10f}")
print(f"Absolute error: {abs(gpu_global_min - cpu_global_min):.3e}")
print(f"Elapsed time: {elapsed:.2f} seconds")

CSV files detected:
  /kaggle/input/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml/random_numbers.csv (4.95 GB)
Using CSV: /kaggle/input/datasets/mehedihasand1497/10-million-random-number-dataset-for-ml/random_numbers.csv
Using 1 feature column(s).
Chunk rows: 100,000


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Processed chunk    1: rows=100,000, values=100,000, current_min=0.0000110989
Processed chunk   10: rows=1,000,000, values=1,000,000, current_min=0.0000001595
Processed chunk   20: rows=2,000,000, values=2,000,000, current_min=0.0000001595
Processed chunk   30: rows=3,000,000, values=3,000,000, current_min=0.0000001480
Processed chunk   40: rows=4,000,000, values=4,000,000, current_min=0.0000001480
Processed chunk   50: rows=5,000,000, values=5,000,000, current_min=0.0000001480
Processed chunk   60: rows=6,000,000, values=6,000,000, current_min=0.0000001480
Processed chunk   70: rows=7,000,000, values=7,000,000, current_min=0.0000001480
Processed chunk   80: rows=8,000,000, values=8,000,000, current_min=0.0000001480
Processed chunk   90: rows=9,000,000, values=9,000,000, current_min=0.0000001480
Processed chunk  100: rows=10,000,000, values=10,000,000, current_min=0.0000001480

DS1 Parallel Reduction complete.
Rows processed: 10,000,000
Values processed: 10,000,000
GPU global minimum: 0